In [29]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

In [30]:
load_dotenv()

True

In [31]:
model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=1.0,
    max_retries=2,
)

In [32]:
# Create the state

class LLMState(TypedDict):
    
    question: str
    answer: str

In [33]:
def llm_qa(state: LLMState) -> LLMState:
    
    # Extract the question from the state
    question = state['question']
    
    # Form a prompt
    prompt = f'Answer the following question: {question}'
    
    # Ask the question to the model
    answer = model.invoke(prompt).content
    
    # Update the answer in the state
    state['answer'] = answer
    
    return state

In [34]:
# create the graph

graph = StateGraph(LLMState)

# Add Nodes
graph.add_node('llm_qa', llm_qa)

# Add Edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# Compile the graph
workflow = graph.compile()

In [35]:
# Execute the graph
initial_state = { 'question': 'What is the capital of India, and explain the reason behind it?' }

final_state = workflow.invoke(initial_state)

print(final_state)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}